# Module 23 — Week 12 — Bayesian Black-Box Optimisation Capstone

**W11: 1/8 improved (F7 ⭐ 3.1034 NEW BEST). F5 plateau confirmed (3651.353 vs best 3651.37). F2 dropped big (0.466 — very noisy). F8 continued momentum (9.9750).**

## Week 12 Strategy Summary
| F | Strategy | Reasoning |
|---|----------|----------|
| F1 | NEAR-EXACT | Keep x2=0.654 (peak); x2=0.670 is a slight probe |
| F2 | GP-EXPLOIT | Push x2 higher (0.942) — W11 drop suggests noise, try higher |
| F3 | W1-ANCHOR | Lock near W1 best (x2=0.969 → 0.954 slight variation) |
| F4 | GP-TRUST | W4 region, GP with recent data |
| F5 | RIDGE-PROBE | x3=0.963 variation (probing ridge shape) |
| F6 | GP-EXPLOIT | Near W8 best with slight x4 adjustment |
| F7 | GP-EXPLOIT | Near W11 best (3.1034) |
| F8 | TREND-EXTRAP | W10→W11 trajectory continued |

In [ ]:
# ── Cell 1: Load all tools ────────────────────────────────────────────────────

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm, yeojohnson
from scipy.stats.qmc import LatinHypercube

import warnings, os
warnings.filterwarnings('ignore')

PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-23/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

print('All libraries loaded — Week 12 ready!')

In [ ]:
# ── Cell 2: All historical submissions and results (W1 to W11) ────────────────

submitted_x_w1 ={1:[0.020584,0.969910],2:[0.814691,0.969505],3:[0.376075,0.370839,0.474761],4:[0.369789,0.452786,0.367951,0.448446],5:[0.241041,0.805036,0.948951,0.905090],6:[0.466959,0.356875,0.489683,0.726384,0.125125],7:[0.027698,0.531762,0.337094,0.176133,0.361503,0.730849],8:[0.192432,0.183093,0.018724,0.036362,0.690267,0.444236,0.081374,0.428967]}
new_y_w1       ={1:1.966e-321,2:0.1292261555216582,3:-0.010707313301147062,4:-0.34595283782499875,5:1450.9433021815964,6:-0.3611823990070205,7:1.4058168801082682,8:9.8915570907296}

submitted_x_w2 ={1:[0.591837,0.591837],2:[0.000000,1.000000],3:[0.421053,1.000000,1.000000],4:[0.909548,0.568955,0.762175,0.811807],5:[0.204881,0.877830,0.879582,0.870578],6:[0.851439,0.906254,0.506372,0.594105,0.708147],7:[0.097054,0.432660,0.338116,0.122619,0.296117,0.886436],8:[0.076274,0.101214,0.383035,0.338493,0.113685,0.882235,0.615428,0.796463]}
new_y_w2       ={1:0.00028209052469858225,2:0.1709619176069506,3:-0.48304244384724265,4:-26.59459580774249,5:1192.2995655092311,6:-1.9259411859252866,7:1.2030170341293975,8:9.0382459830856}

submitted_x_w3 ={1:[0.980000,0.980000],2:[1.000000,0.306122],3:[1.000000,0.000000,0.684211],4:[0.985601,0.686679,0.243615,0.798556],5:[0.204881,0.877830,0.879582,0.870578],6:[0.061416,0.762464,0.106527,0.271402,0.782742],7:[0.067189,0.412831,0.295130,0.070570,0.412599,0.616173],8:[0.682757,0.427203,0.591529,0.734064,0.514947,0.813984,0.722156,0.615073]}
new_y_w3       ={1:2.665897212344236e-174,2:-0.042550557700427774,3:-0.1840890683677661,4:-26.07041694623693,5:1192.2995655092311,6:-2.508952125110497,7:1.2533263563752521,8:7.5792591902086}

submitted_x_w4 ={1:[0.278296,0.020000],2:[0.685269,0.947006],3:[0.403468,0.441923,0.497061],4:[0.352971,0.651614,0.805417,0.616108],5:[0.167299,0.881015,0.978872,0.954244],6:[0.334649,0.293944,0.500782,0.769829,0.074923],7:[0.206363,0.281987,0.389442,0.281544,0.218827,0.711599],8:[0.095545,0.327238,0.051339,0.269531,0.555763,0.417489,0.285113,0.613881]}
new_y_w4       ={1:-2.6647756688938686e-133,2:0.14705786268424045,3:-0.022992940111015336,4:-0.1283640964538999,5:2496.347187728138,6:-0.38592078647528016,7:2.6705394912160187,8:9.8967959631939}

submitted_x_w5 ={1:[0.580092,0.683225],2:[0.702813,0.926626],3:[0.365086,0.316421,0.471038],4:[0.344700,0.645505,0.791987,0.622463],5:[0.139557,0.911522,0.979905,0.977049],6:[0.417831,0.356959,0.468069,0.668531,0.039515],7:[0.384097,0.122113,0.444891,0.357064,0.147383,0.783086],8:[0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246]}
new_y_w5       ={1:0.00008112997850454906,2:0.5833602539566602,3:-0.018707796769607724,4:-13.979947691578896,5:2941.854350298978,6:-0.29985775692426564,7:1.660798293687705,8:9.9275118625839}

submitted_x_w6 ={1:[0.653384,0.652924],2:[0.704856,0.921380],3:[0.151659,0.826046,0.622768],4:[0.291709,0.714786,0.911879,0.664486],5:[0.102387,0.951309,0.977662,0.978193],6:[0.394360,0.399099,0.411100,0.595076,0.020599],7:[0.027785,0.208441,0.270801,0.266138,0.195016,0.698660],8:[0.029320,0.285338,0.223021,0.041430,0.625367,0.719031,0.033888,0.797446]}
new_y_w6       ={1:0.0880341468685302,2:0.6478060146282238,3:-0.0867832750698683,4:-21.252967893168208,5:3303.918327634855,6:-0.5181323257010382,7:2.1498244177996053,8:9.8116383300479}

submitted_x_w7 ={1:[0.533625,0.533688],2:[0.702665,0.919352],3:[0.267218,0.472728,0.513126],4:[0.341374,0.485949,0.606273,0.478470],5:[0.071951,0.977037,0.979767,0.979593],6:[0.441360,0.279908,0.514886,0.684556,0.020780],7:[0.255243,0.272333,0.253655,0.238536,0.237243,0.658362],8:[0.306263,0.307104,0.109816,0.361473,0.669399,0.417361,0.175248,0.274400]}
new_y_w7       ={1:3.8800114757386434e-13,2:0.6475701405048025,3:-0.038391302132802174,4:-4.940966055787715,5:3626.8315773030813,6:-0.33902767001927475,7:2.5915976846312665,8:9.8171020976195}

submitted_x_w8 ={1:[0.643797,0.704343],2:[0.703797,0.924514],3:[0.100454,0.240875,0.185908],4:[0.383806,0.511512,0.656499,0.491038],5:[0.044074,0.979912,0.978237,0.978721],6:[0.409346,0.360704,0.502905,0.718263,0.020264],7:[0.138618,0.329866,0.325614,0.264255,0.295279,0.651123],8:[0.182189,0.037358,0.268170,0.170921,0.585250,0.468337,0.247759,0.632930]}
new_y_w8       ={1:-0.00011412865302137135,2:0.6409162660536529,3:-0.14148000083888568,4:-7.1627894437582,5:3632.183153202294,6:-0.2037089488415273,7:2.7440435657471656,8:9.887359571982}

submitted_x_w9 ={1:[0.651500,0.654000],2:[0.706000,0.921000],3:[0.020793,0.975360,0.381751],4:[0.354000,0.650000,0.806000,0.617000],5:[0.027000,0.980000,0.979500,0.979000],6:[0.420721,0.410509,0.539518,0.760163,0.022635],7:[0.194674,0.230232,0.307303,0.258261,0.265579,0.680946],8:[0.090500,0.066000,0.182000,0.330000,0.768000,0.650000,0.175000,0.502000]}
new_y_w9       ={1:0.09715493565789202,2:0.5975456023703872,3:-0.05325899029920575,4:-14.466436970145782,5:3651.372623492115,6:-0.28461851247025494,7:2.9422351452635813,8:9.9270291}

submitted_x_w10={1:[0.651000,0.654500],2:[0.700060,0.924659],3:[0.020095,0.979354,0.499543],4:[0.335575,0.634933,0.768825,0.615668],5:[0.010000,0.980000,0.979500,0.979000],6:[0.404053,0.356085,0.497149,0.734820,0.020541],7:[0.220332,0.220286,0.350162,0.295013,0.265686,0.636790],8:[0.078337,0.106249,0.150970,0.288837,0.789655,0.619188,0.210082,0.532216]}
new_y_w10      ={1:0.09596,2:0.62631,3:-0.04305,4:-13.086,5:3651.356,6:-0.31774,7:3.0391,8:9.9616}

submitted_x_w11={1:[0.652000,0.654000],2:[0.705000,0.921000],3:[0.030972,0.973585,0.482788],4:[0.346845,0.657931,0.818143,0.620946],5:[0.005000,0.980000,0.979500,0.979000],6:[0.420004,0.371261,0.528732,0.691954,0.026764],7:[0.235451,0.238127,0.353968,0.271570,0.299798,0.643136],8:[0.104788,0.133086,0.121157,0.265760,0.760264,0.590202,0.229279,0.538927]}
new_y_w11      ={1:0.09114,2:0.4656,3:-0.0514,4:-15.239,5:3651.353,6:-0.2783,7:3.1034,8:9.9750}

# All-time bests as of W11
all_time_best = {
    1: (0.09715493565789202, 'W9'),
    2: (0.6478060146282238,  'W6'),
    3: (-0.010707313301147062, 'W1'),
    4: (-0.1283640964538999,   'W4'),
    5: (3651.372623492115,     'W9'),
    6: (-0.2037089488415273,   'W8'),
    7: (3.1034,                'W11'),
    8: (9.9750,                'W11'),
}

print('Historical data loaded W1-W11')
for i in range(1, 9):
    v, w = all_time_best[i]
    print(f'  F{i}: {v:.6f} ({w})')

In [ ]:
# ── Cell 3: Load initial data files + stack all 11 weekly submissions ─────────

BASE_PATH = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',   2: 'Noisy ML Model',
    3: 'Drug Discovery',        4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)', 6: 'Cake Recipe',
    7: 'ML Hyperparameters',    8: 'Complex 8D'
}

data = {}

for i in range(1, 9):
    X_initial = np.load(f'{BASE_PATH}function_{i}/initial_inputs.npy')
    Y_initial = np.load(f'{BASE_PATH}function_{i}/initial_outputs.npy')

    X_all = np.vstack([
        X_initial,
        np.array(submitted_x_w1[i]).reshape(1, -1),
        np.array(submitted_x_w2[i]).reshape(1, -1),
        np.array(submitted_x_w3[i]).reshape(1, -1),
        np.array(submitted_x_w4[i]).reshape(1, -1),
        np.array(submitted_x_w5[i]).reshape(1, -1),
        np.array(submitted_x_w6[i]).reshape(1, -1),
        np.array(submitted_x_w7[i]).reshape(1, -1),
        np.array(submitted_x_w8[i]).reshape(1, -1),
        np.array(submitted_x_w9[i]).reshape(1, -1),
        np.array(submitted_x_w10[i]).reshape(1, -1),
        np.array(submitted_x_w11[i]).reshape(1, -1),
    ])

    Y_all = np.concatenate([
        Y_initial,
        [new_y_w1[i]],  [new_y_w2[i]],  [new_y_w3[i]],
        [new_y_w4[i]],  [new_y_w5[i]],  [new_y_w6[i]],
        [new_y_w7[i]],  [new_y_w8[i]],  [new_y_w9[i]],
        [new_y_w10[i]], [new_y_w11[i]],
    ])

    data[i] = {'X': X_all, 'Y': Y_all}

print('Data loaded for all 8 functions (W1-W11 = 11 weekly points each)!')
for i in range(1, 9):
    n = len(data[i]['Y'])
    v, w = all_time_best[i]
    print(f'  F{i} ({descriptions[i]}):  {n} pts  |  best = {v:.4f} ({w})')

In [ ]:
# ── Cell 4: Helper functions ──────────────────────────────────────────────────

LOWER_BOUND = 0.02
UPPER_BOUND = 0.98

def transform_outputs(Y, method='log'):
    if method == 'log':
        return np.log(np.abs(Y) + 1e-300) * np.sign(Y + 1e-300)
    elif method == 'yeojohnson':
        Y_transformed, _ = yeojohnson(Y)
        return Y_transformed
    else:
        return Y.copy()

def expected_improvement(mu, sigma, current_best, xi=0.01):
    improvement = mu - current_best - xi
    Z  = improvement / (sigma + 1e-9)
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def analyse_w12(func_num, beta, trust_center, trust_radius,
                xi=0.01, alpha=1e-6, y_transform='log', policy='exploit'):

    X   = data[func_num]['X']
    Y   = data[func_num]['Y']
    dim = X.shape[1]

    best_idx = np.argmax(Y)
    best_Y   = Y[best_idx]

    print(f'\n{"="*65}')
    print(f'F{func_num} — {descriptions[func_num]} ({dim}D)')
    print(f'Policy       : {policy.upper()}')
    print(f'Data points  : {len(Y)}')
    print(f'All-time best: {best_Y:.6f}')
    print(f'W11 result   : {new_y_w11[func_num]:.6f}')
    print(f'Trust center : {np.round(trust_center, 4).tolist()}')
    print(f'Trust radius : {trust_radius}   Beta : {beta}')
    print('='*65)

    sorted_pairs = sorted(zip(Y, range(len(Y))), reverse=True)
    print(f'\n  Top 5 observations:')
    for rank, (y_val, idx) in enumerate(sorted_pairs[:5]):
        marker = '  <- ALL-TIME BEST' if rank == 0 else ''
        x_str  = ', '.join([f'{v:.4f}' for v in X[idx]])
        print(f'  [{rank+1}]  Y = {y_val:+.4f}   X = [{x_str}]{marker}')

    Y_transformed = transform_outputs(Y, method=y_transform)

    box_lo = np.clip(trust_center - trust_radius, LOWER_BOUND, UPPER_BOUND)
    box_hi = np.clip(trust_center + trust_radius, LOWER_BOUND, UPPER_BOUND)

    sampler      = LatinHypercube(d=dim, seed=42)
    raw_samples  = sampler.random(n=50000)
    X_candidates = box_lo + raw_samples * (box_hi - box_lo)

    print(f'\n  Generated 50,000 LHS candidates inside trust region')

    kernel = Matern(length_scale=0.2, nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=alpha,
        n_restarts_optimizer=3, normalize_y=True
    )
    gp.fit(X, Y_transformed)

    mu, sigma = gp.predict(X_candidates, return_std=True)

    ucb_scores   = mu + beta * sigma
    best_ucb_idx = np.argmax(ucb_scores)

    current_best_t = float(transform_outputs(np.array([best_Y]), method=y_transform)[0])
    ei_scores    = expected_improvement(mu, sigma, current_best_t, xi=xi)
    best_ei_idx  = np.argmax(ei_scores)

    mu_ucb = float(gp.predict(X_candidates[best_ucb_idx].reshape(1,-1)).ravel()[0])
    mu_ei  = float(gp.predict(X_candidates[best_ei_idx].reshape(1,-1)).ravel()[0])

    next_x = X_candidates[best_ei_idx] if mu_ei >= mu_ucb else X_candidates[best_ucb_idx]
    winner = 'EI' if mu_ei >= mu_ucb else 'UCB'
    print(f'  GP winner   : {winner}  (UCB_mean={mu_ucb:.4f}, EI_mean={mu_ei:.4f})')

    # Duplicate check against all W1-W11 submissions
    prior = [
        np.array(submitted_x_w1[func_num]),  np.array(submitted_x_w2[func_num]),
        np.array(submitted_x_w3[func_num]),  np.array(submitted_x_w4[func_num]),
        np.array(submitted_x_w5[func_num]),  np.array(submitted_x_w6[func_num]),
        np.array(submitted_x_w7[func_num]),  np.array(submitted_x_w8[func_num]),
        np.array(submitted_x_w9[func_num]),  np.array(submitted_x_w10[func_num]),
        np.array(submitted_x_w11[func_num]),
    ]
    min_d = min(np.linalg.norm(next_x - p) for p in prior)
    print(f'  Min dist from prior : {min_d:.4f}')

    if min_d < 0.015:
        np.random.seed(99)
        next_x = np.clip(next_x + np.random.uniform(-0.02, 0.02, dim), LOWER_BOUND, UPPER_BOUND)
        print(f'  WARNING: Too close — nudged away')

    portal_string = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> GP candidate F{func_num}: {portal_string} <<<')
    return next_x, portal_string

print('Helper functions ready!')

---
## F1 — Radiation Detection (2D)
**W11 result: 0.09114 ❌ Below W9 best (0.09715)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | NEAR-EXACT | x2=0.654 confirmed peak; slight probe at x2=0.670 |
| Trust center | W9 best (0.6515, 0.654) | All-time best |
| Trust radius | 0.02 | Very tight — spike very narrow in x2 |

**W11 lesson:** x2=0.654 is narrow peak. Probing x2=0.670 to test whether spike has width.

In [ ]:
W9_BEST_F1 = np.array([0.651500, 0.654000])

next_x1, portal1 = analyse_w12(
    func_num=1, beta=0.3,
    trust_center=W9_BEST_F1, trust_radius=0.02,
    xi=0.001, y_transform='log', policy='near-exact'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Probe x2=0.670 (Δ=+0.016 from peak) while keeping x1 near W9 best.
# GP confirms x1=0.651500 stays optimal; x2 probe tests spike width.
portal1 = '0.651500-0.670000'
next_x1 = np.array([0.651500, 0.670000])
print()
print('  MANUAL OVERRIDE: x1=0.6515 (W9 best) | x2=0.670 probe (Δ+0.016 from peak)')
print(f'  Dist from W9: {np.linalg.norm(next_x1 - W9_BEST_F1):.4f}')
print(f'  Final submission F1: {portal1}')

---
## F2 — Noisy ML Model (2D)
**W11 result: 0.4656 ❌ Big drop — function is very noisy**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | GP-EXPLOIT | Push x2 higher (0.942) — W11 was 0.921, try higher side |
| Trust center | W6 best (0.704856, 0.921380) | Confirmed highest ever |
| Trust radius | 0.03 | Small |

**W11 analysis:** W6/W7 gave ~0.647-0.648. W11 at x2=0.921 dropped to 0.466 — highly stochastic.
Strategy: push x2 to 0.942 to test if higher x2 is better.

In [ ]:
W6_BEST_F2 = np.array([0.704856, 0.921380])

next_x2, portal2 = analyse_w12(
    func_num=2, beta=0.5,
    trust_center=W6_BEST_F2, trust_radius=0.03,
    xi=0.005, y_transform='log', policy='gp-exploit'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Push x2 to 0.942 — W10 tried 0.924 (0.626), W11 tried 0.921 (0.466 — noisy).
# Testing x2=0.942 to explore whether true peak is above 0.921.
# x1=0.704856 held fixed at W6 optimal.
portal2 = '0.704856-0.942000'
next_x2 = np.array([0.704856, 0.942000])
print()
print('  MANUAL OVERRIDE: x1=0.704856 (W6 best) | x2=0.942 probe (above W6 optimal)')
print(f'  Dist from W6: {np.linalg.norm(next_x2 - W6_BEST_F2):.4f}')
print(f'  Final submission F2: {portal2}')

---
## F3 — Drug Discovery (3D)
**W11 result: -0.0514 ❌ Still failing to match W1 best (-0.011)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | W1-ANCHOR | Return very close to W1 exact coordinates |
| Trust center | W1 best (0.020584, 0.969910, 0.474761) | Only point that gave -0.011 |
| Trust radius | 0.05 | Small |

**Pattern:** W1 (x2=0.9699) gave -0.011. W9 (x2=0.9754) gave -0.053. W11 (x2=0.9736) gave -0.051.
Need x2 closer to 0.954 to probe slightly different angle. x3=0.475 confirmed good.

In [ ]:
W1_BEST_F3 = np.array([0.020584, 0.969910, 0.474761])

next_x3, portal3 = analyse_w12(
    func_num=3, beta=1.0,
    trust_center=W1_BEST_F3, trust_radius=0.05,
    xi=0.001, y_transform='log', policy='w1-anchor'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Try x2=0.954 (slightly below W1's 0.9699) — test if there's a better angle.
# x1=0.020 (same as W1), x3=0.475 (confirmed good from W1).
# W11 had x2=0.9736 → -0.051. W1 had x2=0.9699 → -0.011. Moving x2 down slightly.
portal3 = '0.020000-0.954000-0.475000'
next_x3 = np.array([0.020000, 0.954000, 0.475000])
print()
print('  MANUAL OVERRIDE: x2=0.954 probe (Δ-0.016 from W1 x2=0.9699)')
print(f'  Dist from W1 best: {np.linalg.norm(next_x3 - W1_BEST_F3):.4f}')
print(f'  Final submission F3: {portal3}')

---
## F4 — Warehouse Placement (4D)
**W11 result: -15.239 ❌ W4 region consistently failing W9-W11**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | GP-TRUST-W4 | GP anchored to W4 best, recent-only data |
| Trust center | W4 best (0.352971, 0.651614, 0.805417, 0.616108) | Only point with -0.128 |
| Trust radius | 0.10 | Moderate — exploring around W4 region |

**W11 analysis:** Consistent degradation W9(-14.5) → W10(-13.1) → W11(-15.2). Non-stationary.

In [ ]:
# Use recent-only data for F4 (non-stationary landscape)
init_X4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_inputs.npy')
init_Y4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_outputs.npy')
clean_mask = init_Y4 > -20

recent_X4 = np.array([
    [0.352971, 0.651614, 0.805417, 0.616108],  # W4:  -0.1284
    [0.341374, 0.485949, 0.606273, 0.478470],  # W7:  -4.94
    [0.383806, 0.511512, 0.656499, 0.491038],  # W8:  -7.163
    [0.354000, 0.650000, 0.806000, 0.617000],  # W9:  -14.466
    [0.335575, 0.634933, 0.768825, 0.615668],  # W10: -13.086
    [0.346845, 0.657931, 0.818143, 0.620946],  # W11: -15.239
])
recent_Y4 = np.array([-0.1284, -4.94, -7.163, -14.466, -13.086, -15.239])

data[4]['X'] = np.vstack([init_X4[clean_mask], recent_X4])
data[4]['Y'] = np.concatenate([init_Y4[clean_mask], recent_Y4])

W4_BEST_F4 = np.array([0.352971, 0.651614, 0.805417, 0.616108])

next_x4, portal4 = analyse_w12(
    func_num=4, beta=1.0,
    trust_center=W4_BEST_F4, trust_radius=0.10,
    xi=0.01, alpha=0.1, y_transform='yeojohnson',
    policy='gp-trust-w4'
)

# ── Manual override ───────────────────────────────────────────────────────────
# GP candidate near W4 region with x2 adjusted slightly higher.
portal4 = '0.337655-0.667407-0.837231-0.628203'
next_x4 = np.array([0.337655, 0.667407, 0.837231, 0.628203])
prior_f4 = [submitted_x_w9[4], submitted_x_w10[4], submitted_x_w11[4]]
print()
print('  MANUAL OVERRIDE: GP near W4 region, adjusted by GP UCB winner')
print(f'  Min dist from W9/W10/W11: {min(np.linalg.norm(next_x4-np.array(p)) for p in prior_f4):.4f}')
print(f'  Final submission F4: {portal4}')

---
## F5 — Chemical Yield STAR (4D)
**W11 result: 3651.353 ❌ Plateau — W9(3651.37) / W10(3651.356) / W11(3651.353)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | RIDGE-PROBE | Test x3=0.963 variation — does the ridge have different slope? |
| x1 | 0.027000 | Confirmed ridge peak |
| x2/x4 | 0.980 / 0.979 | Confirmed good |
| x3 | 0.963 | Probe: slightly below 0.9795 to test ridge slope in x3 |

**W11 analysis:** x1 at 0.005 (W11) gave same result as x1=0.010 (W10) and x1=0.027 (W9).
The W9 result (3651.37) used x1=0.027. Returning to x1=0.027. Probe x3=0.963.

In [ ]:
W9_BEST_F5 = np.array([0.027000, 0.980000, 0.979500, 0.979000])

next_x5, portal5 = analyse_w12(
    func_num=5, beta=0.02,
    trust_center=W9_BEST_F5, trust_radius=0.015,
    xi=0.01, y_transform='log',
    policy='ridge-probe'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Probe x3=0.963 (below the 0.9795 that gave best W9 result).
# x1=0.027 restored (best W9 result). x2=0.980, x4=0.979 held.
# Testing whether x3 ridge has a wider shape or steeper slope.
portal5 = '0.027000-0.980000-0.963000-0.979000'
next_x5 = np.array([0.027000, 0.980000, 0.963000, 0.979000])
prior_f5 = [submitted_x_w9[5], submitted_x_w10[5], submitted_x_w11[5]]
min_d5 = min(np.linalg.norm(next_x5 - np.array(p)) for p in prior_f5)
print()
print('  MANUAL OVERRIDE: x1=0.027 restored | x3=0.963 probe (Δ-0.0165 from W9 x3)')
print(f'  Min dist from W9/W10/W11: {min_d5:.4f}')
print(f'  Final submission F5: {portal5}')

---
## F6 — Cake Recipe (5D)
**W11 result: -0.2783 ❌ Improving but still below W8 best (-0.2037)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | GP-EXPLOIT | GP near W8 best; adjust x4 toward W8 |
| Trust center | W8 best (0.409346, 0.360704, 0.502905, 0.718263, 0.020264) | All-time best |
| Trust radius | 0.06 | Moderate |

**W11 analysis:** W11 had x4=0.692 → -0.278. W8 had x4=0.718 → -0.204. x4 needs to be higher.
Return to W8 region with slight GP adjustment.

In [ ]:
W8_BEST_F6 = np.array([0.409346, 0.360704, 0.502905, 0.718263, 0.020264])

next_x6, portal6 = analyse_w12(
    func_num=6, beta=0.3,
    trust_center=W8_BEST_F6, trust_radius=0.06,
    xi=0.001, y_transform='log',
    policy='gp-exploit'
)

# ── Manual override ───────────────────────────────────────────────────────────
# GP anchored near W8 best. x4=0.701 (between W8's 0.718 and W11's 0.692).
# x5=0.022 (kept near W8's 0.020264 — critical for minimising output).
portal6 = '0.417095-0.367584-0.502073-0.701098-0.022008'
next_x6 = np.array([0.417095, 0.367584, 0.502073, 0.701098, 0.022008])
prior_f6 = [submitted_x_w8[6], submitted_x_w9[6], submitted_x_w10[6], submitted_x_w11[6]]
print()
print('  MANUAL OVERRIDE: GP near W8 best | x4=0.701 (testing between W8/W11 x4 values)')
print(f'  Min dist from W8/W9/W10/W11: {min(np.linalg.norm(next_x6-np.array(p)) for p in prior_f6):.4f}')
print(f'  Final submission F6: {portal6}')

---
## F7 — ML Hyperparameters (6D)
**W11 result: 3.1034 ✅ NEW BEST — 6 consecutive improvements!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | GP-EXPLOIT | Near W11 best (3.1034) — continue momentum |
| Trust center | W11 best (0.235451, 0.238127, 0.353968, 0.271570, 0.299798, 0.643136) | New all-time best |
| Trust radius | 0.05 | Tight — exploit confirmed momentum |

**W11 momentum:** W9(2.94) → W10(3.04) → W11(3.10). GP with tight trust region to continue.

In [ ]:
W11_BEST_F7 = np.array([0.235451, 0.238127, 0.353968, 0.271570, 0.299798, 0.643136])

next_x7, portal7 = analyse_w12(
    func_num=7, beta=0.5,
    trust_center=W11_BEST_F7, trust_radius=0.05,
    xi=0.01, y_transform='log',
    policy='gp-exploit'
)

# ── Manual override ───────────────────────────────────────────────────────────
# GP candidate near W11 best. Continue positive trend.
# x6 pushed slightly (0.660 vs W11's 0.643) to test direction of improvement.
portal7 = '0.242071-0.235871-0.349962-0.303448-0.305843-0.659759'
next_x7 = np.array([0.242071, 0.235871, 0.349962, 0.303448, 0.305843, 0.659759])
print()
print('  MANUAL OVERRIDE: GP near W11 best | x6=0.660 (Δ+0.017 from W11 x6=0.643)')
print(f'  Dist from W11 best: {np.linalg.norm(next_x7 - W11_BEST_F7):.4f}')
print(f'  Final submission F7: {portal7}')

---
## F8 — Complex 8D
**W11 result: 9.9750 ✅ NEW BEST — W10→W11 trend continues!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | TREND-EXTRAP | W10→W11 trajectory extrapolated one more step |
| Trust center | W11 best (0.104788, ...) | New all-time best |

**Trend per dimension (W10→W11→W12 extrapolation):**
- x1: 0.078→0.105→**0.132** (+0.027/step)
- x2: 0.106→0.133→**0.171** (+0.038/step)
- x3: 0.151→0.121→**0.124** (roughly stable)
- x4: 0.289→0.266→**0.228** (−0.038/step)
- x5: 0.790→0.760→**0.795** (oscillating ~0.775-0.795)
- x6: 0.619→0.590→**0.555** (−0.035/step)
- x7: 0.210→0.229→**0.264** (+0.035/step)
- x8: 0.532→0.539→**0.565** (+0.026/step)

In [ ]:
W11_BEST_F8 = np.array([0.104788, 0.133086, 0.121157, 0.265760,
                         0.760264, 0.590202, 0.229279, 0.538927])

next_x8, portal8 = analyse_w12(
    func_num=8, beta=0.8,
    trust_center=W11_BEST_F8, trust_radius=0.10,
    xi=0.01, y_transform='log',
    policy='trend-extrapolation'
)

# ── Manual override: trend extrapolation from W11 new best ────────────────────
# W10→W11 produced 2 consecutive all-time bests.
# Each dimension extrapolated one step from W10→W11 trajectory.
portal8 = '0.131852-0.171318-0.123876-0.228240-0.795114-0.554833-0.264222-0.564700'
next_x8 = np.array([0.131852, 0.171318, 0.123876, 0.228240,
                    0.795114, 0.554833, 0.264222, 0.564700])
print()
print('  STRATEGY: Trend extrapolation W10→W11→W12 per dimension')
print(f'  Dist from W11 best: {np.linalg.norm(next_x8-W11_BEST_F8):.4f}')
print(f'  Final submission F8: {portal8}')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────

print('=' * 70)
print('WEEK 12 — MODULE 23 — PORTAL SUBMISSION STRINGS')
print('=' * 70)
print()

all_portals = {1:portal1, 2:portal2, 3:portal3, 4:portal4,
               5:portal5, 6:portal6, 7:portal7, 8:portal8}

all_strategies = {
    1: 'NEAR-EXACT    x1=0.6515 | x2=0.670 probe (Δ+0.016 from peak)',
    2: 'GP-EXPLOIT    x1=0.7049 | x2=0.942 probe (above W6 optimal)',
    3: 'W1-ANCHOR     x2=0.954 probe | x1=0.020 | x3=0.475',
    4: 'GP-TRUST-W4   GP near W4 region, recent-only data, Yeo-Johnson',
    5: 'RIDGE-PROBE   x1=0.027 | x3=0.963 (Δ-0.017 from W9 x3)',
    6: 'GP-EXPLOIT    Near W8 best | x4=0.701 (between W8/W11 x4)',
    7: 'GP-EXPLOIT    Near W11 best | x6=0.660 (Δ+0.017 from W11)',
    8: 'TREND-EXTRAP  W10→W11 trajectory continued per dimension',
}

for i in range(1, 9):
    best_v, best_w = all_time_best[i]
    w11_v = new_y_w11[i]
    direction = '↑' if w11_v > best_v else ('✓' if abs(w11_v-best_v)<0.001 else '↓')
    print(f'F{i}: {all_portals[i]}')
    print(f'     Strategy : {all_strategies[i]}')
    print(f'     W11 result: {w11_v:.4f} {direction}  |  All-time best: {best_v:.4f} ({best_w})')
    print()

print('=' * 70)
print('COPY-PASTE READY:')
print('=' * 70)
for i in range(1, 9):
    print(f'F{i}: {all_portals[i]}')

In [ ]:
# ── Plot: Full 11-week progress for all 8 functions ───────────────────────────

weekly_y = {
    1: new_y_w1,  2: new_y_w2,  3: new_y_w3,  4: new_y_w4,
    5: new_y_w5,  6: new_y_w6,  7: new_y_w7,  8: new_y_w8,
    9: new_y_w9, 10: new_y_w10, 11: new_y_w11,
}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('BBO Capstone W1-W11 Full Progress — Module 23 Week 12 Prep', fontsize=14, fontweight='bold')

for idx, fn in enumerate(range(1, 9)):
    ax    = axes[idx // 4][idx % 4]
    weeks = list(range(1, 12))
    vals  = [weekly_y[w][fn] for w in weeks]
    running_best = [max(vals[:w]) for w in range(1, len(vals)+1)]

    ax.plot(weeks, vals, 'o--', color='steelblue', alpha=0.6, label='Weekly query')
    ax.plot(weeks, running_best, 's-', color='darkorange', linewidth=2, label='Running best')

    best_v, best_w = all_time_best[fn]
    w11_v = new_y_w11[fn]
    ax.set_title(
        f'F{fn}: {descriptions[fn]}\nbest={best_v:.4f} ({best_w})  W11={w11_v:.4f}',
        fontsize=7.5
    )
    ax.set_xlabel('Week')
    ax.set_ylabel('Output')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)
    ax.axvline(x=11, color='red', linestyle=':', alpha=0.5, label='W11 (latest)')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'w11_full_progress.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Progress plot saved: {plot_path}')